In [0]:
storage_account = "dlforformula1"
container = "bronze"
client_id = "16e41178-4311-45a7-b2ec-ff3999a7593b"
tenant_id = "0bf495b2-f9e4-4786-82ea-d1ff2c6f9353"
account_fqdn = f"{storage_account}.dfs.core.windows.net"

# (Optional but helpful) remove any key-based config that may be set on cluster
for k in [
    f"fs.azure.account.key.{account_fqdn}",
    f"fs.azure.account.keyprovider.{account_fqdn}",
    "fs.azure.account.key"
]:
    try:
        spark.conf.unset(k)
    except:
        pass

# Set OAuth configs (IMPORTANT: include the account FQDN in the key)
spark.conf.set(f"fs.azure.account.auth.type.{account_fqdn}", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{account_fqdn}",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{account_fqdn}", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{account_fqdn}",
               dbutils.secrets.get(scope="adls-scope", key="client-secret"))
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{account_fqdn}",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

In [0]:
df = (spark.read
        .format("csv")
        .options(
            header=True,
            inferSchema=True,
            delimiter=","
        )
        .load("abfss://demo@dlforformula1.dfs.core.windows.net/circuits.csv"))
# display(df)
print(df.columns)
print(df.dtypes)
df.printSchema()

In [0]:
## Apply schema
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType,
    DoubleType, DateType
)

schema = StructType([
    StructField("circuitId", IntegerType(), True),
    StructField("circuitRef", StringType(), True),
    StructField("name", StringType(), True),
    StructField("location", StringType(), True),
    StructField("country", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lng", DoubleType(), True),
    StructField("alt", IntegerType(), True),
    StructField("url", StringType(), True)
])

In [0]:
df = (spark.read
        .schema(schema)
        .option("header", "true")
        .csv("abfss://demo@dlforformula1.dfs.core.windows.net/circuits.csv"))
df.printSchema()

In [0]:
from pyspark.sql.functions import col
## selecting needed columns- 5 ways
temp_df = df.select("circuitId", "circuitRef", "name", "location", "country", "lat", "lng", "alt")
temp_df.printSchema()
# column style
temp_df1 = df.select(df.circuitId, df.circuitRef, df.name, df.location, df.country, df.lat, df.lng, df.alt)
temp_df1.printSchema()
# column style
from pyspark.sql.functions import col, upper
temp_df2 = df.select(
    col("circuitId"),
    col("circuitRef"),
    col("name").alias("circuit_name"),
    col("location"),
    upper(col("country")).alias("country"),
    col("lat")
)
temp_df2.printSchema()
# array style
temp_df3 = df.select(df["circuitId"], df["circuitRef"], df["name"], df["location"], df["country"], df["lat"])
temp_df3.printSchema()
# sql-style
temp_df4 = df.selectExpr("circuitId", "circuitRef", "name", "location", "country", "lat")
temp_df4.printSchema()

In [0]:
## Drop columns:
temp_df = df.drop("url")
# temp_df.printSchema()

## drop multiple columns
# dataframes are immutable -> we have to assign the result to a new dataframe
df2 = df.drop("lat", "lng", "alt")
# df2.printSchema()

## method 3
cols_to_keep = [c for c in df.columns if c != "url"]
df2 = df.select(*cols_to_keep)
# df2.printSchema()

## Sql method of drpping
df.createOrReplaceTempView("circuits") ## This registers the DataFrame as a SQL table inside Spark.
# SELECT circuitId, circuitRef, name, location, country FROM circuits
df2 = spark.sql("""
SELECT circuitId, circuitRef, name, location, country
FROM circuits
""")
df2.printSchema()

In [0]:
## withColumnRenamed
df2 = df.withColumnRenamed("circuitId", "circuit_id")
df2.printSchema()

## withColumn()
df2 = df.withColumn()